# 🕵️ Dia 18 — Monte seu laboratório de investigação e mapeie a topologia

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 18* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-18.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Sábado, 10/out. A hospedagem do site falso foi suspensa na sexta, e uma cópia do IscaKit chegou à Vigia por meio da cooperativa. A Marta quer que vocês "deem uma olhada".

Rafa: *"Não no seu notebook, não na rede da Vigia e não com internet. Kit de phishing chama de volta para casa. Duas máquinas, uma rede que não vai a lugar nenhum, e uma prova escrita de que não vai."*

Neste notebook você vai:

1. **Conferir o `day18-verify.txt`** contra os itens do checkpoint, automaticamente.
2. **Conferir as conversas da captura** contra `10.66.0.0/24` com o módulo `ipaddress`.
3. **Gerar o mapa de topologia** em Graphviz e Mermaid a partir dos seus valores.
4. **Montar o `build-log/README.md`** com as regras do laboratório e a quarentena do IscaKit.
5. **Jogar o quiz de fechamento da Fase 1**.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo funciona **sem internet**; quando uma célula tenta a rede, ela tem plano B embutido.

## 1. O `day18-verify.txt` passa no checkpoint?

Se você tem o seu arquivo, suba-o no Colab (ícone de pasta) com o nome `day18-verify.txt`. Senão, a célula usa uma saída de exemplo com o formato exato do Passo 5.

In [ ]:
import re
from pathlib import Path

EXEMPLO_VERIFY = '2026-10-10T14:02:31Z\nlo               UNKNOWN        127.0.0.1/8 ::1/128\nenp0s3           UP             10.66.0.10/24 fe80::a00:27ff:fe4e:66a1/64\n10.66.0.0/24 dev enp0s3 proto kernel scope link src 10.66.0.10 metric 100\nPING 10.66.0.20 (10.66.0.20) 56(84) bytes of data.\n64 bytes from 10.66.0.20: icmp_seq=1 ttl=64 time=0.412 ms\n64 bytes from 10.66.0.20: icmp_seq=2 ttl=64 time=0.388 ms\n64 bytes from 10.66.0.20: icmp_seq=3 ttl=64 time=0.401 ms\n--- 10.66.0.20 ping statistics ---\n3 packets transmitted, 3 received, 0% packet loss, time 2031ms\nStarting Nmap 7.94SVN ( https://nmap.org ) at 2026-10-10 14:02 UTC\nNmap scan report for 10.66.0.10\nHost is up.\nNmap scan report for 10.66.0.20\nHost is up (0.00041s latency).\nNmap done: 256 IP addresses (2 hosts up) scanned in 2.07 seconds\nStarting Nmap 7.94SVN ( https://nmap.org ) at 2026-10-10 14:02 UTC\nNmap scan report for 10.66.0.20\nHost is up (0.00038s latency).\nPORT   STATE SERVICE\n22/tcp open  ssh\n80/tcp open  http\nNmap done: 1 IP address (1 host up) scanned in 0.04 seconds\nHTTP/1.1 200 OK\nServer: nginx/1.24.0 (Ubuntu)\nDate: Sat, 10 Oct 2026 14:02:40 GMT\nping: connect: Network is unreachable\nISOLATION OK: no route to internet\n;; UDP setup with 1.1.1.1#53(1.1.1.1) for example.com failed: network unreachable.\nISOLATION OK: no DNS to internet\n'

meu = Path("day18-verify.txt")
if meu.is_file():
    texto, fonte = meu.read_text(encoding="utf-8", errors="replace"), "o SEU day18-verify.txt"
else:
    texto, fonte = EXEMPLO_VERIFY, "saída de exemplo (suba a sua para valer)"
print("Fonte:", fonte, "\n")

# Cada item do checkpoint vira uma regra que procura no texto
CHECAGENS = [
    ("Ping bem-sucedido para 10.66.0.20",
     lambda t: re.search(r"3 packets transmitted, 3 received, 0% packet loss", t) is not None),
    ("Porta 22 aberta em 10.66.0.20",  lambda t: re.search(r"^22/tcp\s+open", t, re.M) is not None),
    ("Porta 80 aberta em 10.66.0.20",  lambda t: re.search(r"^80/tcp\s+open", t, re.M) is not None),
    ("Sem rota padrão (nenhuma linha 'default')",
     lambda t: re.search(r"^default\b", t, re.M) is None),
    ("Falha registrada ao alcançar 1.1.1.1 (ping)",
     lambda t: "ISOLATION OK: no route to internet" in t),
    ("Falha registrada ao alcançar 1.1.1.1 (DNS)",
     lambda t: "ISOLATION OK: no DNS to internet" in t),
    ("Carimbo de hora UTC no topo",
     lambda t: re.match(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z", t.strip()) is not None),
]
resultado_verify = {}
for nome, regra in CHECAGENS:
    ok = regra(texto)
    resultado_verify[nome] = ok
    print(("✅ " if ok else "❌ ") + nome)
hora_verificacao = texto.strip().splitlines()[0]

## 2. A captura só tem tráfego do laboratório?

O Passo 9 pede para olhar *Statistics > Conversations* no Wireshark. Exporte como CSV (*Copy > as CSV*) e suba como `conversas.csv`, ou use a saída de exemplo.

In [ ]:
import csv, io, ipaddress

EXEMPLO_CONV = 'Address A,Address B,Packets,Bytes\n10.66.0.10,10.66.0.20,58,6120\nfe80::a00:27ff:fe4e:66a1,ff02::2,3,210\nfe80::a00:27ff:fe9b:20b4,ff02::2,2,140\n'
LAB = ipaddress.ip_network("10.66.0.0/24")

conv_path = Path("conversas.csv")
conv_txt = conv_path.read_text() if conv_path.is_file() else EXEMPLO_CONV

def fora_do_lab(endereco):
    """True se o endereço NÃO é permitido na captura do laboratório."""
    ip = ipaddress.ip_address(endereco)
    if ip.is_multicast:                       # ff02::2 (descoberta de roteador IPv6) é só local
        return False
    if ip.version == 6:
        return not ip.is_link_local            # IPv6: só fe80::
    return ip not in LAB                       # IPv4: só 10.66.0.x

problemas = []
for linha in csv.DictReader(io.StringIO(conv_txt)):
    for lado in ("Address A", "Address B"):
        if fora_do_lab(linha[lado]):
            problemas.append(linha[lado])
    print(f"{linha['Address A']:<28} ↔ {linha['Address B']:<12} {linha['Packets']:>4} pacotes")

alcancou_1111 = "1.1.1.1" in conv_txt
print()
print("✅ só endereços do laboratório" if not problemas else f"❌ fora do laboratório: {sorted(set(problemas))}")
print("✅ nenhum pacote para 1.1.1.1" if not alcancou_1111 else "❌ houve conversa com 1.1.1.1")
isolamento_ok = all(resultado_verify.values()) and not problemas and not alcancou_1111

> 💡 Este teste é o Dia 9 voltando: `ip in rede` com CIDR. Guarde a função junto com o laboratório e rode-a depois de **todo** exercício que gerar captura. O isolamento que você provou hoje deixa de valer no dia em que alguém conectar o NAT e esquecer.

## 3. O mapa a partir dos seus valores

Preencha com a saída dos **seus** comandos (`VBoxManage --version`, `ip -br link`, `uname -m`). Os valores de exemplo são marcadores e aparecem entre parênteses.

In [ ]:
#@title Valores do seu laboratório { display-mode: "form" }
autor = "(seu nome)"  #@param {type:"string"}
so_host = "(SO do host)"  #@param {type:"string"}
versao_vbox = "(versão do 00-virtualbox-version.txt)"  #@param {type:"string"}
arquitetura = "(x86_64 ou arm64)"  #@param {type:"string"}
mac_analyst = "(MAC do inv-analyst)"  #@param {type:"string"}
mac_target = "(MAC do inv-target)"  #@param {type:"string"}
material_de_caso_presente = True  #@param {type:"boolean"}

aviso_nat = "Material de caso presente: NAT proibido" if material_de_caso_presente else "Sem material de caso: NAT só para atualização registrada"

DOT = f"""digraph invlab {{
  compound=true; rankdir=TB; fontname="Helvetica";
  label="topologia do invlab, versão 1, {autor}, {hora_verificacao}"; labelloc=t;
  node [shape=box, style=rounded, fontname="Helvetica"];
  subgraph cluster_host {{
    label="Host: {so_host}, VirtualBox {versao_vbox}, {arquitetura}";
    subgraph cluster_lab {{
      label="Laboratório isolado (fronteira de confiança) - verificado {hora_verificacao}"; style=dashed; color=red;
      net [shape=underline, label="invlab 10.66.0.0/24\\nsem gateway, sem DHCP"];
      analyst [label="inv-analyst (analista)\\nUbuntu Desktop 24.04 LTS, 2 vCPU, 4 GB\\nenp0s3 10.66.0.10/24, {mac_analyst}\\nWireshark, tcpdump, nmap\\ncaptura: enp0s3\\nsnapshot: clean-baseline"];
      target  [label="inv-target (alvo)\\nUbuntu Server 24.04 LTS, 1 vCPU, 2 GB\\nenp0s3 10.66.0.20/24, {mac_target}\\n22/tcp ssh, 80/tcp http\\nsnapshot: clean-baseline"];
      analyst -> net [dir=none]; target -> net [dir=none];
    }}
  }}
  internet [shape=ellipse, label="Internet"];
  analyst -> internet [style=dashed, label="NAT (nic2)\\ndesconectado por padrão\\n{aviso_nat}"];
  legenda [shape=note, label="Legenda\\nlinha cheia: caminho ativo\\ntracejada: só em atualização registrada\\nvermelho tracejado: fronteira do laboratório"];
}}"""

Path("notas").mkdir(exist_ok=True)
Path("notas/invlab.dot").write_text(DOT, encoding="utf-8")
print(DOT)
print("\n💾 notas/invlab.dot — renderize com: dot -Tpng invlab.dot -o invlab-topology.png")

In [ ]:
# Opcional: o Colab tem Graphviz? Linhas com "!" são puladas pelo teste.
!dot -Tpng notas/invlab.dot -o notas/invlab-topology.png && echo "PNG gerado em notas/"
print("(Se o Graphviz não estiver disponível, cole o .dot em qualquer visualizador de Graphviz.)")

## 4. O `build-log/README.md` com as regras e a quarentena

Os *hashes* dos arquivos do `build-log/` entram se você subi-los para uma pasta `build-log/` aqui. As regras e a quarentena do IscaKit (README, Passo 10) vêm do formulário.

In [ ]:
#@title Regras do laboratório e quarentena { display-mode: "form" }
regra_1 = "Aqui rodam varreduras, capturas e material suspeito, só entre inv-analyst e inv-target."  #@param {type:"string"}
regra_2 = "Nada daqui é apontado para fora do invlab: nem nmap, nem o kit, nem uma requisição."  #@param {type:"string"}
regra_3 = "Depois de cada exercício que altera uma VM, restauro o snapshot clean-baseline."  #@param {type:"string"}
regra_4 = "O NAT não é conectado enquanto houver material de caso em qualquer VM."  #@param {type:"string"}
entrada_1 = "sha256sum iscakit.zip > iscakit.zip.sha256   # no host, ANTES de entrar"  #@param {type:"string"}
entrada_2 = "sha256sum -c iscakit.zip.sha256              # na VM, DEPOIS de entrar (pasta compartilhada somente leitura)"  #@param {type:"string"}
entrada_3 = "python scripts/registrar_evidencia.py iscakit.zip \"IscaKit recebido via Serra Azul\""  #@param {type:"string"}
espera_1 = "Consulta DNS ou conexão HTTPS para a API de um bot de mensageiro, levando as credenciais"  #@param {type:"string"}
espera_2 = "Envio de e-mail (SMTP) com os dados capturados para uma caixa do operador"  #@param {type:"string"}
espera_3 = "Requisição a um painel ou contador remoto, ou checagem do IP de quem está vendo"  #@param {type:"string"}

import hashlib
linhas_hash = []
pasta_log = Path("build-log")
if pasta_log.is_dir():
    for p in sorted(pasta_log.iterdir()):
        if p.is_file() and p.name != "README.md":
            linhas_hash.append(f"| `{p.name}` | `{hashlib.sha256(p.read_bytes()).hexdigest()}` |")
tabela = "\n".join(linhas_hash) or "| (suba os arquivos do build-log/ para calcular) | |"

readme = f"""# Registro de montagem do invlab

Verificação de isolamento (UTC): {hora_verificacao} — {"aprovada" if isolamento_ok else "PENDENTE: ver notebook"}

## Arquivos

| Arquivo | SHA-256 |
|---|---|
{tabela}

## Períodos com o NAT conectado
| Início (UTC) | Fim (UTC) | Motivo |
|---|---|---|
| | | |

## Snapshots
- inv-target: clean-baseline
- inv-analyst: clean-baseline

## Regras deste laboratório
1. {regra_1}
2. {regra_2}
3. {regra_3}
4. {regra_4}

## Quarentena do IscaKit (Operação Isca)
### Entrada de material
```
{entrada_1}
{entrada_2}
{entrada_3}
```
### O que esperamos que o kit tente (e nada disso pode sair do invlab)
- {espera_1}
- {espera_2}
- {espera_3}

Nenhuma dessas conexões pode sair: seria acessar sistema alheio (Dia 1, checagem 3) e avisaria o operador de que o kit está sendo analisado, e de onde.
"""
Path("notas/build-log-README.md").write_text(readme, encoding="utf-8")
Path("notas/isca-dia18-quarentena.md").write_text(readme[readme.index("## Quarentena"):], encoding="utf-8")
print(readme)

## 5. 🎮 Quiz de fechamento da Fase 1

Uma pergunta por dia, do 9 ao 18. Responda com a letra.

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("Dia 9: 10.66.0.20 está em 10.66.0.0/24?", ["a) Sim", "b) Não"], "a"),
    ("Dia 10: o touch -d consegue voltar o ctime para 2019?", ["a) Sim", "b) Não, o ctime vai para agora"], "b"),
    ("Dia 11: 14 Referers do site falso no log do site real significam...", ["a) 14 vítimas", "b) pelo menos 13 navegadores abriram a página falsa"], "b"),
    ("Dia 12: login fora da linha de base com sudo de manutenção depois. A avaliação...", ["a) 'conta comprometida, confirmado'", "b) traz a hipótese concorrente e o fato que a derrubaria"], "b"),
    ("Dia 13: por que diffar o dns.stable.txt?", ["a) porque o TTL muda a cada rodada", "b) porque é menor"], "a"),
    ("Dia 14: FAILED em 4 arquivos depois do logrotate prova adulteração?", ["a) Sim", "b) Não: comparar por conteúdo, prefixo e compactação"], "b"),
    ("Dia 15: no golpe do Pix, a entrada no internet banking é...", ["a) T1190", "b) T1078"], "b"),
    ("Dia 16: o CPF no formulário de denúncia. Primeira mitigação:", ["a) cifrar", "b) não coletar"], "b"),
    ("Dia 17: a primeira ação do plano de prontidão fecha...", ["a) o repúdio", "b) a negação de serviço"], "a"),
    ("Dia 18: quando conectar o NAT do inv-analyst com o IscaKit numa VM?", ["a) para atualizar rapidinho", "b) nunca, enquanto houver material de caso"], "b"),
]
acertos = 0
for i, (p, ops, certa) in enumerate(QUIZ, start=1):
    print(f"\n[{i:02d}/{len(QUIZ)}] {p}")
    for o in ops:
        print("   ", o)
    if MODO_JOGO:
        r = ""
        while r not in ("a", "b"):
            r = input("   Resposta (a/b): ").strip().lower()
        acertos += (r == certa)
        print("   ✅" if r == certa else f"   ❌ Era {certa}.")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}", "— Fase 1 concluída. 🥷 Rank D consolidado." if acertos >= 8 else "— Meta: 8. Releia os dias em que errou.")

## 6. 📥 Baixar

In [ ]:
try:
    from google.colab import files
    for f in ("invlab.dot", "build-log-README.md", "isca-dia18-quarentena.md"):
        files.download(f"notas/{f}")
except ImportError:
    print("(Fora do Colab: os arquivos ficaram em notas/)")

## ✅ Antes de fechar o Dia 18 (e a Fase 1)

- [ ] Quiz de fechamento com pelo menos 8/10
- [ ] Os 4 itens do `day18-verify.txt` aprovados com o **seu** arquivo
- [ ] Captura conferida: só `10.66.0.x` e `fe80::`, nenhum `1.1.1.1`
- [ ] `invlab-topology.png` com os 9 elementos e a linha "Material de caso presente: NAT proibido"
- [ ] `build-log/README.md` com hashes, períodos de NAT, snapshots, regras e a quarentena do IscaKit
- [ ] Os 18 cartões da Fase 1 em **Artefato construído**
- [ ] `status` do dia 18 = `estudado` no `curriculo/mapa-dos-dias.csv`

**Amanhã (Dia 19), Fase 2:** o ciclo OSINT completo. A Marta quer "tudo o que esse golpe tem pendurado na internet", e o Rafa vai pedir uma pergunta antes de qualquer aba aberta.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>